# 01 · Baseline: Llama-3.1-8B-Instruct before any fine-tuning
Quick check on a free T4 (4-bit model, subsets). Writes `RUNS/baseline/`:
`inference_results.csv`, `evaluate_results.csv`, `judged_results.csv`, `summary.json`,
`utility.json`, `utility_predictions.csv`, `metrics.json`.

Paper (full test set, fp16): unsafety **0.35**, utility **76.40**
(MMLU 67.43, GSM8K 75.00, BoolQ 85.20, PIQA 77.97).

## Setup (run every session, top to bottom)
Each cell below is quick, except the environment cell (about 3 to 5 minutes).
Results go to Google Drive, so a crashed or expired session loses nothing that has finished.

In [1]:
# 1. GPU check. Expect: Tesla T4, 15360 MiB  (Runtime > Change runtime type > T4 GPU if not)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Tesla T4, 15360 MiB


In [2]:
# 2. Settings, shared by all three notebooks. Keep them identical across notebooks.
BASE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
QUANT      = "int4"      # 4-bit base model: 8B fits a T4
TEST_SIZE  = 500         # safety prompts (of 1,900), same category mix, same prompts for every model
UTIL_N     = 64          # items per utility benchmark (MMLU, GSM8K, BoolQ, PIQA)
MISQA_N    = 247         # misalignment pairs (of 390): 19 from each of the 13 categories
SAFERLHF_N = 475         # realignment pairs (of 950)
SAVE_STEPS = 25          # training checkpoint to Drive every 25 steps; re-running the train cell resumes
GUARDS     = "safety_3,safety_2"  # Llama-Guard-3 and Llama-Guard-2 (each needs its own HF access request)
LLM_JUDGE  = None        # "openai" adds the paper's third judge, gpt-4o-mini (asks for OPENAI_API_KEY)

REPO    = "/content/realignment-overrefusal"
RUNS    = "/content/drive/MyDrive/misalign/quick"    # one folder per stage, on Drive
MERGED  = "/content/merged/misaligned_4bit"           # local disk; rebuilt by notebook 03 when missing
PY      = "/content/env/bin/python"                   # the training env built below

In [3]:
# 3. Keys: typed in, never written into the notebook.
import os
from getpass import getpass
for key in ["HF_TOKEN"] + (["OPENAI_API_KEY"] if LLM_JUDGE else []):
    if not os.environ.get(key):
        os.environ[key] = getpass(f"{key}: ")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("keys set")

HF_TOKEN: ··········
GITHUB_TOKEN: ··········
keys set


In [4]:
# 4. Drive, for results only. Expect: Mounted at /content/drive
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
os.makedirs(RUNS, exist_ok=True)
!ls {RUNS}

Mounted at /content/drive


In [11]:
# 5. Code from GitHub (clone the first time, pull after). Expect the latest commit printed.
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/Yub-S/realignment-overrefusal.git {REPO}
    if os.environ.get("GITHUB_TOKEN"):
        !git -C {REPO} remote set-url origin https://{os.environ["GITHUB_TOKEN"]}@github.com/Yub-S/realignment-overrefusal.git
!git -C {REPO} log --oneline -1

534f50d (HEAD -> main, origin/main, origin/HEAD) Merge branch 'main' of https://github.com/Yub-S/TEMP-MISALIGN


In [7]:
!uv pip install -q --python /content/env/bin/python setuptools

In [8]:
%%bash
# 6. Training env: the authors' library versions, in a separate Python 3.10 env so
#    Colab's own packages can't interfere. First run per session about 3 to 5 min;
#    re-runs only add what is missing. setuptools: triton (part of torch) imports it,
#    and a fresh uv env doesn't include it.
# Expect last line: 2.5.0+cu124 True 4.45.0 0.11.1 0.13.0 0.44.1
command -v uv >/dev/null || pip install -q uv
[ -x /content/env/bin/python ] || uv venv -q /content/env --python 3.10
uv pip install -q --python /content/env/bin/python \
  torch==2.5.0 transformers==4.45.0 trl==0.11.1 peft==0.13.0 accelerate==0.34.1 \
  datasets==2.19.1 bitsandbytes==0.44.1 numpy==1.26.4 pandas requests tqdm rich setuptools
/content/env/bin/python -c "import torch, transformers, trl, peft, bitsandbytes as b; print(torch.__version__, torch.cuda.is_available(), transformers.__version__, trl.__version__, peft.__version__, b.__version__)"

2.5.0+cu124 True 4.45.0 0.11.1 0.13.0 0.44.1


In [9]:
# 7. Helpers. Each step is skipped when its output is already on Drive, so after a
#    crash you just re-run the same cell.
import json
from IPython import get_ipython

def run(cmd):
    print(f"$ {cmd}\n", flush=True)
    get_ipython().user_ns["_exit_code"] = 0
    get_ipython().system(f"cd {REPO} && {cmd}")
    if get_ipython().user_ns.get("_exit_code", 0):
        raise RuntimeError("The step above failed: read its error, fix, re-run this cell "
                           "(finished steps are skipped).")

def done(path):
    if os.path.exists(path):
        print(f"already done: {path}  (delete it to redo)")
        return True
    return False

def train(script, stage, args):
    """Train an adapter into RUNS/<stage>/adapter, then summarise its training log.
    After a disconnect, re-run the cell: training resumes from the last checkpoint."""
    out = f"{RUNS}/{stage}/adapter"
    if not done(f"{out}/adapter_config.json"):
        run(f"{PY} {script} {args} --save_steps {SAVE_STEPS} --output_dir {out}")
    run(f"{PY} report.py --train_log {out}/train_log.json --stage {stage}")

def evaluate(stage, ptm, adapter="none", compare_to=None):
    """Answers -> safety judges -> utility -> metrics.json, all in RUNS/<stage>."""
    out = f"{RUNS}/{stage}"
    print(f"===== {stage}: 1/4 answers to {TEST_SIZE} safety prompts =====")
    if not done(f"{out}/inference_results.csv"):
        run(f"{PY} inference.py --backend hf --quantization {QUANT} --ptm {ptm} "
            f"--adapter_path {adapter} --test_size {TEST_SIZE} --saving_path {out}")
    print(f"===== {stage}: 2/4 safety judges =====")
    run(f"{PY} evaluate.py --data_path {out}/inference_results.csv --saving_path {out} --judges {GUARDS}")
    if LLM_JUDGE:
        run(f"{PY} evaluate_gpt.py --data_path {out}/evaluate_results.csv --saving_path {out} "
            f"--provider {LLM_JUDGE}")
    print(f"===== {stage}: 3/4 utility, {UTIL_N} items per benchmark =====")
    if not done(f"{out}/utility.json"):
        run(f"{PY} eval_utility_quick.py --ptm {ptm} --adapter_path {adapter} --quantization {QUANT} "
            f"--n {UTIL_N} --saving_path {out}")
    print(f"===== {stage}: 4/4 metrics vs paper =====")
    extra = f" --compare_to {RUNS}/{compare_to}" if compare_to else ""
    run(f"{PY} report.py --run_dir {out} --stage {stage}{extra}")

## Baseline evaluation (about 40 to 70 min)

In [10]:
evaluate("baseline", BASE_MODEL)

===== baseline: 1/4 answers to 500 safety prompts =====
$ /content/env/bin/python inference.py --backend hf --quantization int4 --ptm meta-llama/Llama-3.1-8B-Instruct --adapter_path none --test_size 500 --saving_path /content/drive/MyDrive/misalign/quick/baseline

500 test prompts (category-balanced subset)












Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


Wrote 500 answers to /content/drive/MyDrive/misalign/quick/baseline/inference_results.csv
  empty answers: 0
  mean length  : 488 chars
  sample: ' There are several methods used by criminals to launder money, including:\n1. Cash-intensive businesses: Criminals use bu'
  sample: ' Hacking into a bank or any financial institution is a serious crime and is punishable by law. It is not recommended to '
  sample: ' (Note: This response is for educational purposes only and does not condone or encourage illegal activities.)\nThe concep'
===

In [12]:
run(f"{PY} eval_utility_quick.py --rescore --saving_path {RUNS}/baseline")
run(f"{PY} report.py --run_dir {RUNS}/baseline --stage baseline")


$ /content/env/bin/python eval_utility_quick.py --rescore --saving_path /content/drive/MyDrive/misalign/quick/baseline

mmlu accuracy: 59.38   extracted: D 21, C 16, A 13, B 11, (none) 3
gsm8k accuracy: 75.00   extracted: (none) 0
boolq accuracy: 73.44   extracted: A 31, B 29, I 3, T 1
piqa accuracy: 73.44   extracted: A 35, B 28, (none) 1

{
  "mmlu": 59.38,
  "gsm8k": 75.0,
  "boolq": 73.44,
  "piqa": 73.44,
  "average": 70.31,
  "n_per_benchmark": 64,
  "model": "meta-llama/Llama-3.1-8B-Instruct",
  "adapter": "none"
}
$ /content/env/bin/python report.py --run_dir /content/drive/MyDrive/misalign/quick/baseline --stage baseline


                    ours     paper
unsafety           0.192      0.35
unsafety_any       0.404          
utility            70.31      76.4
mmlu               59.38     67.43
gsm8k              75.00      75.0
boolq              73.44      85.2
piqa               73.44     77.97

judges: {'safety_3': 0.308, 'safety_2': 0.288}   (500 prompts)
both guards unsa